<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Predicción de Aterrizaje de la Primera Etapa del Falcon 9 de Space X**[cite: 2]


## Extracción web (Web scraping) de los Registros de Lanzamientos de Falcon 9 y Falcon Heavy desde Wikipedia[cite: 2]


Tiempo estimado necesario: **40** minutos[cite: 2]


En este laboratorio, realizará extracción web (web scraping) para recopilar registros históricos de lanzamientos del Falcon 9 desde una página de Wikipedia titulada `List of Falcon 9 and Falcon Heavy launches`[cite: 2]

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches[cite: 2]


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


La primera etapa del Falcon 9 aterrizará con éxito[cite: 2]


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Varios ejemplos de un aterrizaje fallido se muestran aquí:[cite: 2]


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


Más específicamente, los registros de lanzamiento se almacenan en una tabla HTML que se muestra a continuación:[cite: 2]


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


## Objetivos[cite: 2]
Extraer registros de lanzamientos de Falcon 9 con `BeautifulSoup`:[cite: 2]
- Extraer una tabla HTML de registros de lanzamiento de Falcon 9 desde Wikipedia[cite: 2]
- Analizar (parsear) la tabla y convertirla en un marco de datos (data frame) de Pandas[cite: 2]


Primero importemos los paquetes requeridos para este laboratorio[cite: 2]


In [ ]:
!pip3 install beautifulsoup4
!pip3 install requests

In [ ]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

y le proporcionaremos algunas funciones auxiliares para procesar la tabla HTML extraída de la web[cite: 2]


In [ ]:
def date_time(table_cells):
    """
    Esta función devuelve la fecha y hora desde la celda de la tabla HTML
    Entrada: el elemento de una celda de datos de tabla extrae la fila adicional
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    Esta función devuelve la versión del propulsor desde la celda de la tabla HTML 
    Entrada: el elemento de una celda de datos de tabla extrae la fila adicional
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    Esta función devuelve el estado de aterrizaje desde la celda de la tabla HTML
    Entrada: el elemento de una celda de datos de tabla extrae la fila adicional
    """
    out=[i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass

def extract_column_from_header(row):
    """
    Esta función devuelve el estado de aterrizaje desde la celda de la tabla HTML 
    Entrada: el elemento de una celda de datos de tabla extrae la fila adicional
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filtrar el dígito y nombres vacíos
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


Para mantener las tareas del laboratorio consistentes, se le pedirá que extraiga los datos de una captura de la página Wiki `List of Falcon 9 and Falcon Heavy launches` actualizada el `9th June 2021`[cite: 2]


In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

A continuación, solicite la página HTML de la URL anterior y obtenga un objeto `response`[cite: 2]


### TAREA 1: Solicitar la página Wiki del Lanzamiento del Falcon9 desde su URL[cite: 2]


Primero, realicemos un método HTTP GET para solicitar la página HTML de lanzamientos del Falcon9, como una respuesta HTTP.[cite: 2]


In [ ]:
# use el método requests.get() con el static_url y headers proporcionados
# asigne la respuesta a un objeto
response = requests.get(static_url, headers=headers)

Cree un objeto `BeautifulSoup` a partir de la `response` HTML[cite: 2]


In [ ]:
# Use BeautifulSoup() para crear un objeto BeautifulSoup a partir del contenido de texto de la respuesta
soup = BeautifulSoup(response.text, 'html.parser')

Imprima el título de la página para verificar si el objeto `BeautifulSoup` se creó correctamente[cite: 2]


In [ ]:
# Use el atributo soup.title
print(soup.title)

### TAREA 2: Extraer todos los nombres de columnas/variables del encabezado de la tabla HTML[cite: 2]


A continuación, queremos recopilar todos los nombres de las columnas relevantes desde el encabezado de la tabla HTML[cite: 2]


Intentemos encontrar todas las tablas en la página de Wikipedia primero. Si necesita refrescar su memoria sobre `BeautifulSoup`, consulte el enlace de referencia externo hacia el final de este laboratorio[cite: 2]


In [ ]:
# Use la función find_all en el objeto BeautifulSoup, con el tipo de elemento `table`
# Asigne el resultado a una lista llamada `html_tables`
html_tables = soup.find_all('table')

A partir de la tercera tabla se encuentra nuestra tabla objetivo, que contiene los registros de lanzamiento reales.[cite: 2]


In [ ]:
# Imprimamos la tercera tabla y verifiquemos su contenido
first_launch_table = html_tables[2]
print(first_launch_table)

Debería poder ver los nombres de las columnas incrustados en los elementos `<th>` del encabezado de la tabla de la siguiente manera:[cite: 2]


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


A continuación, solo necesitamos iterar a través de los elementos `<th>` y aplicar la función proporcionada `extract_column_from_header()` para extraer el nombre de la columna uno por uno[cite: 2]


In [ ]:
column_names = []

# Aplique la función find_all() con el elemento `th` en first_launch_table
# Itere cada elemento th y aplique extract_column_from_header() proporcionada para obtener el nombre de la columna
# Agregue el nombre de columna no vacío (`if name is not None and len(name) > 0`) en una lista llamada column_names
for row in first_launch_table.find_all('th'):
    name = extract_column_from_header(row)
    if name is not None and len(name) > 0:
        column_names.append(name)

Compruebe los nombres de las columnas extraídas[cite: 2]


In [ ]:
print(column_names)

## TAREA 3: Crear un marco de datos (data frame) analizando las tablas HTML de lanzamiento[cite: 2]


Crearemos un diccionario vacío con claves a partir de los nombres de las columnas extraídas en la tarea anterior. Más adelante, este diccionario se convertirá en un dataframe de Pandas.[cite: 2]


In [ ]:
launch_dict= dict.fromkeys(column_names)

# Eliminar una columna irrelevante
del launch_dict['Date and time ( )']

# Inicialicemos el launch_dict con cada valor como una lista vacía
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Agregar algunas columnas nuevas
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

A continuación, solo necesitamos completar `launch_dict` con los registros de lanzamiento extraídos de las filas de la tabla.[cite: 2]


Por lo general, las tablas HTML en las páginas Wiki pueden contener anotaciones inesperadas y otros tipos de ruidos, como enlaces de referencia `B0004.1[8]`, valores faltantes `N/A [e]`, formato inconsistente, etc.[cite: 2]


Para simplificar el proceso de análisis, hemos proporcionado un fragmento de código incompleto a continuación para ayudarlo a completar `launch_dict`. Complete el siguiente fragmento de código con los TODOs o puede optar por escribir su propia lógica para analizar todas las tablas de lanzamiento:[cite: 2]


In [ ]:
extracted_row = 0
#Extraer cada tabla 
for table_number,table in enumerate(soup.find_all('table',"wikitable plainrowheaders collapsible")):
   # obtener la fila de la tabla 
    for rows in table.find_all("tr"):
        #comprobar si el primer encabezado de la tabla es un número correspondiente a un lanzamiento 
        if rows.th:
            if rows.th.string:
                flight_number=rows.th.string.strip()
                flag=flight_number.isdigit()
        else:
            flag=False
        #obtener el elemento de la tabla 
        row=rows.find_all('td')
        #si es un número guardar las celdas en un diccionario 
        if flag:
            extracted_row += 1
            # Valor del número de vuelo
            # TODO: Agregar el flight_number a launch_dict con la clave `Flight No.`
            launch_dict['Flight No.'].append(flight_number)
            
            datatimelist=date_time(row[0])
            
            # Valor de la fecha
            # TODO: Agregar la fecha a launch_dict con la clave `Date`
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            
            # Valor de la hora
            # TODO: Agregar la hora a launch_dict con la clave `Time`
            time = datatimelist[1]
            launch_dict['Time'].append(time)
              
            # Versión del propulsor
            # TODO: Agregar el bv a launch_dict con la clave `Version Booster`
            bv=booster_version(row[1])
            if not(bv):
                bv=row[1].a.string if row[1].a else row[1].text.strip()
            launch_dict['Version Booster'].append(bv)
            
            # Sitio de lanzamiento
            # TODO: Agregar el sitio de lanzamiento a launch_dict con la clave `Launch site`
            launch_site = row[2].a.string if row[2].a else row[2].text.strip()
            launch_dict['Launch site'].append(launch_site)
            
            # Carga útil (Payload)
            # TODO: Agregar el payload a launch_dict con la clave `Payload`
            payload = row[3].a.string if row[3].a else row[3].text.strip()
            launch_dict['Payload'].append(payload)
            
            # Masa de la carga útil
            # TODO: Agregar el payload_mass a launch_dict con la clave `Payload mass`
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            
            # Órbita
            # TODO: Agregar el orbit a launch_dict con la clave `Orbit`
            orbit = row[5].a.string if row[5].a else row[5].text.strip()
            launch_dict['Orbit'].append(orbit)
            
            # Cliente
            # TODO: Agregar el customer a launch_dict con la clave `Customer`
            customer = row[6].a.string if row[6].a else row[6].text.strip()
            launch_dict['Customer'].append(customer)
            
            # Resultado del lanzamiento
            # TODO: Agregar el launch_outcome a launch_dict con la clave `Launch outcome`
            launch_outcome = list(row[7].strings)[0] if list(row[7].strings) else ""
            launch_dict['Launch outcome'].append(launch_outcome)
            
            # Aterrizaje del propulsor
            # TODO: Agregar el booster_landing a launch_dict con la clave `Booster landing`
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)


Después de haber rellenado los valores analizados del registro de lanzamiento en `launch_dict`, puede crear un dataframe a partir de él.[cite: 2]


In [ ]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

Ahora podemos exportarlo a un **CSV** para la siguiente sección, pero para que las respuestas sean consistentes y en caso de que tenga dificultades para terminar este laboratorio.[cite: 2]

Los laboratorios siguientes utilizarán un conjunto de datos proporcionado para hacer que cada laboratorio sea independiente.[cite: 2]


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


## Autores[cite: 2]


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Registro de Cambios
-->


<!--
| Fecha (AAAA-MM-DD) | Versión | Cambiado Por | Descripción del Cambio      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Actualizaciones de Tareas   |
| 2020-11-10        | 1.0     | Nayef      | Creó la versión inicial     |
-->


Derechos de autor © 2021 IBM Corporation. Todos los derechos reservados.[cite: 2]
